# 03 - Test the Connected Model Through APIM

Run [02_configure.ipynb](02_configure.ipynb) first. Use **Python (byom01-capability-testing)** with the working directory set to `notebooks/`.

**[PREVIEW]** Upstream Responses routing must be available in this Foundry project. The signed-in user needs **Foundry User** on the project and log-query access to the configured workspace. Service role assignments may require propagation time.

**Paid inference:** running **Invoke Connected Model** makes one Foundry Responses request. It does not call APIM or the backend directly. Run only with approval.

A unique, harmless test marker travels in the prompt. APIM records only that marker, routing identifiers, and status—not request/response bodies. A model reply alone does not prove the gateway route.

To recheck an existing run in a fresh kernel, run **Load Gateway Settings** and **Check Gateway Evidence** only. Do not rerun **Create Test Agent** or **Invoke Connected Model**. The ignored evidence file keeps the latest test.

## Load Gateway Settings

In [ ]:
import json
from datetime import datetime, timedelta, timezone
from pathlib import Path
from uuid import uuid4

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import AzureCliCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus

demo_root = Path("..").resolve()
gateway = json.loads((demo_root / "env" / "gateway.json").read_text(encoding="utf-8"))
evidence_path = demo_root / "env" / "test-evidence.json"
credential = AzureCliCredential(subscription=gateway["subscriptionId"])
project_client = AIProjectClient(endpoint=gateway["projectEndpoint"], credential=credential)
openai_client = project_client.get_openai_client(max_retries=0)
logs_client = LogsQueryClient(credential)
print(f"Project: {gateway['projectId']}")
print(f"Connected model: {gateway['modelReference']}")

## Create Test Agent

Create a uniquely named prompt agent with no tools. Its model is the registered `<connection-name>/<deployment-name>` reference. The previous test must be cleaned up before replacing its evidence.

In [ ]:
if evidence_path.exists():
    previous = json.loads(evidence_path.read_text(encoding="utf-8"))
    if not previous["agentVersionDeleted"] or not previous["conversationDeleted"]:
        raise RuntimeError("Run Clean Up Test Resources before starting another test.")

test_id = str(uuid4())
agent = project_client.agents.create_version(
    agent_name=f"byom01-test-{test_id}",
    definition=PromptAgentDefinition(
        model=gateway["modelReference"],
        instructions="Reply briefly to the connectivity check. Do not use tools.",
    ),
)
evidence = {
    "testId": test_id,
    "subscriptionId": gateway["subscriptionId"],
    "projectId": gateway["projectId"],
    "projectEndpoint": gateway["projectEndpoint"],
    "modelReference": gateway["modelReference"],
    "apiId": gateway["apiId"],
    "backendResourceId": gateway["backendResourceId"],
    "backendUrl": gateway["backendUrl"],
    "deploymentName": gateway["deploymentName"],
    "projectPrincipalId": gateway["projectPrincipalId"],
    "workspaceId": gateway["workspaceId"],
    "agentName": agent.name,
    "agentVersion": agent.version,
    "conversationId": None,
    "invokedAt": None,
    "responseId": None,
    "responseStatus": None,
    "outputText": None,
    "gatewayRecords": [],
    "verification": "not-invoked",
    "agentVersionDeleted": False,
    "conversationDeleted": True,
}
evidence_path.write_text(json.dumps(evidence, indent=2) + "\n", encoding="utf-8")
print(f"Test agent: {agent.name}, version {agent.version}")

## Invoke Connected Model

Send one harmless prompt through the consuming Foundry project. Save the invocation result independently of telemetry. The conversation and agent version remain available for inspection until you run **Clean Up Test Resources**; underlying service errors are not caught or reformatted.

Create a new test agent before another invocation. To retry failed deletion, use **Clean Up Test Resources**, not this paid cell.

In [ ]:
evidence = json.loads(evidence_path.read_text(encoding="utf-8"))
assert evidence["projectId"] == gateway["projectId"], "Evidence belongs to another project"
assert not evidence["agentVersionDeleted"], "Create a new test agent before invoking"
assert evidence["invokedAt"] is None, "This test has already been invoked"
conversation = openai_client.conversations.create()
evidence["conversationId"] = conversation.id
evidence["conversationDeleted"] = False
evidence["invokedAt"] = datetime.now(timezone.utc).isoformat()
evidence["verification"] = "pending"
evidence_path.write_text(json.dumps(evidence, indent=2) + "\n", encoding="utf-8")
response = openai_client.responses.create(
    conversation=conversation.id,
    input=f"Connectivity check byom01-test:{evidence['testId']}. Reply 'Gateway connection works'.",
    extra_body={"agent_reference": {
        "name": evidence["agentName"], "type": "agent_reference",
    }},
)
evidence["responseId"] = response.id
evidence["responseStatus"] = response.status
evidence["outputText"] = response.output_text
evidence_path.write_text(json.dumps(evidence, indent=2) + "\n", encoding="utf-8")
assert response.status == "completed", f"Foundry response did not complete: {response.status}"
assert response.output_text.strip(), "Foundry returned no output text"
print(response.output_text)
print(f"Foundry response: {response.id}; test: {evidence['testId']}")
print("Gateway route remains unverified until Check Gateway Evidence succeeds.")
print("Run Clean Up Test Resources when finished inspecting this test.")

## Save Invocation Evidence

The invocation cell writes `env/test-evidence.json` before any telemetry query. It contains this harmless test's output and identifiers, never credentials. Reopening it does not repeat inference.

In [ ]:
evidence = json.loads(evidence_path.read_text(encoding="utf-8"))
print(f"Saved evidence: {evidence_path}")
print(f"Foundry response: {evidence['responseId']}; verification: {evidence['verification']}")

## Clean Up Test Resources

Run this when finished inspecting the test. This is the only cell that deletes test resources. It retries only deletions not recorded as successful; it also cleans up an agent created but never invoked. It targets the saved test's own version/conversation, not another agent or the configured gateway.

Run this before leaving a failed or interrupted test. Cleanup failures propagate and remain visible in the evidence file.

In [ ]:
evidence = json.loads(evidence_path.read_text(encoding="utf-8"))
assert evidence["projectId"] == gateway["projectId"], "Evidence belongs to another project"
try:
    if not evidence["conversationDeleted"]:
        openai_client.conversations.delete(conversation_id=evidence["conversationId"])
        evidence["conversationDeleted"] = True
finally:
    try:
        if not evidence["agentVersionDeleted"]:
            project_client.agents.delete_version(
                agent_name=evidence["agentName"], agent_version=evidence["agentVersion"], force=True,
            )
            evidence["agentVersionDeleted"] = True
    finally:
        evidence_path.write_text(json.dumps(evidence, indent=2) + "\n", encoding="utf-8")
print("Test conversation/version deleted; connection and gateway retained.")

## Check Gateway Evidence

Query the saved run's workspace using its exact test marker. Match the API, requested deployment, fixed backend URL/resource, validated project identity, and a successful backend status. The original Foundry response must also be completed and nonempty.

Telemetry can take several minutes to appear. No matching successful record means **pending/unverified**, not success. Rerun only this cell to recheck; it makes no model call. Query failures and partial results are errors, not verification.

In [ ]:
from uuid import UUID

evidence = json.loads(evidence_path.read_text(encoding="utf-8"))
assert evidence["subscriptionId"] == gateway["subscriptionId"], "Evidence belongs to another subscription"
test_id = str(UUID(evidence["testId"]))
assert evidence["invokedAt"] is not None, "This test has not been invoked"
invoked_at = datetime.fromisoformat(evidence["invokedAt"])
query = f'''
AppTraces
| where Message has "byom01-backend-result"
| extend testId = tostring(Properties["testId"])
| where testId == "{test_id}"
| project ['time'] = TimeGenerated, requestId = tostring(Properties["requestId"]),
    apiId = tostring(Properties["apiId"]), backendResourceId = tostring(Properties["backendResourceId"]),
    backendUrl = tostring(Properties["backendUrl"]), deploymentName = tostring(Properties["deploymentName"]),
    callerPrincipalId = tostring(Properties["callerPrincipalId"]), statusCode = toint(Properties["statusCode"])
| order by ['time'] asc
'''
evidence["verification"] = "pending"
evidence_path.write_text(json.dumps(evidence, indent=2) + "\n", encoding="utf-8")
result = logs_client.query_workspace(
    evidence["workspaceId"], query,
    timespan=(invoked_at - timedelta(minutes=1), datetime.now(timezone.utc)),
)
if result.status != LogsQueryStatus.SUCCESS:
    raise RuntimeError(f"Incomplete gateway log query: {result.partial_error}")
records = [
    dict(zip(table.columns, row))
    for table in result.tables for row in table.rows
]
for record in records:
    record["time"] = record["time"].isoformat()
evidence["gatewayRecords"] = records
successful_records = [record for record in records if
    record["requestId"]
    and record["apiId"] == evidence["apiId"]
    and record["backendResourceId"].lower() == evidence["backendResourceId"].lower()
    and record["backendUrl"] == evidence["backendUrl"]
    and record["deploymentName"] == evidence["deploymentName"]
    and record["callerPrincipalId"].lower() == evidence["projectPrincipalId"].lower()
    and record["statusCode"] is not None and 200 <= record["statusCode"] < 300
]
response_completed = evidence["responseStatus"] == "completed" and bool(evidence["outputText"] and evidence["outputText"].strip())
if successful_records and response_completed:
    evidence["verification"] = "verified"
    print(f"VERIFIED: Foundry response {evidence['responseId']} and APIM request {successful_records[0]['requestId']}")
else:
    print("PENDING/UNVERIFIED: no matching successful gateway record and completed Foundry response yet.")
    print("Recheck this cell after telemetry ingestion; do not repeat inference for a telemetry delay.")
evidence["checkedAt"] = datetime.now(timezone.utc).isoformat()
evidence_path.write_text(json.dumps(evidence, indent=2) + "\n", encoding="utf-8")
print(json.dumps(records, indent=2))

A successful test needs both the Foundry response and matching gateway evidence. The correlation marker is carried in the request payload; the outer Foundry response ID is not assumed to equal a backend ID.

Keep the connection and gateway for later experiments. For full-demo cleanup, follow the [README](../README.md#cleanup) and unchanged destroy notebook. Never delete pre-existing groups as test cleanup.